In [1]:
import pandas as pd

In [2]:
# tidy up esol datasets
# use Delaney's dataset first
# from: ESOL:  Estimating Aqueous Solubility Directly from Molecular Structure
#       John S. Delaney
#       Journal of Chemical Information and Computer Sciences 2004 44 (3), 1000-1005
#       DOI: 10.1021/ci034243x 

In [4]:
delaney = pd.read_csv('esol_delaney.csv')

In [12]:
# check if we can parse all SMILES using rdkit
from rdkit.Chem import MolFromSmiles, MolToSmiles

#delaney['mol'] = delaney.SMILES.apply(MolFromSmiles)
delaney['nonstereo_aromatic_smiles'] = delaney.SMILES.apply(lambda x: MolToSmiles(MolFromSmiles(x), isomericSmiles=False, canonical=True))

In [13]:
# no errors, so we can continue

In [14]:
delaney.shape

(1144, 6)

In [15]:
delaney.nonstereo_aromatic_smiles.drop_duplicates().shape

(1115,)

In [16]:
delaney = delaney.drop_duplicates(subset='nonstereo_aromatic_smiles')

In [20]:
delaney = delaney.rename(columns={'measured log(solubility:mol/L)': 'solubility'})

In [22]:
delaney = delaney[['nonstereo_aromatic_smiles', 'solubility']]

In [23]:
delaney

,nonstereo_aromatic_smiles,solubility
0,ClCC(Cl)(Cl)Cl,-2.180
1,CC(Cl)(Cl)Cl,-2.000
2,ClC(Cl)C(Cl)Cl,-1.740
3,ClCC(Cl)Cl,-1.480
4,FC(F)(Cl)C(F)(Cl)Cl,-3.040
...,...,...
1139,CNC(=O)C(C)SCCSP(=O)(OC)OC,1.144
1140,C=CC1(C)OC(=O)N(c2cc(Cl)cc(Cl)c2)C1=O,-4.925
1141,CC(=O)CC(c1ccccc1)c1c(O)c2ccccc2oc1=O,-3.893
1142,Cc1cccc(C)c1NC(=O)c1cc(S(N)(=O)=O)c(Cl)cc1O,-3.790


In [24]:
delaney.to_csv('regression/esol_delaney.csv', index=False)

In [25]:
# now work on Mobley's database
# v 0.51
# https://escholarship.org/uc/item/6sd403pz
# from: Mobley, D.L., Guthrie, J.P. FreeSolv: a database of experimental and calculated hydration free energies, with input files. J Comput Aided Mol Des 28, 711–720 (2014). https://doi.org/10.1007/s10822-014-9747-x

In [30]:
mobley = pd.read_csv('mobley_database.txt', delimiter=';', header=2)

In [34]:
mobley['nonstereo_aromatic_smiles'] = mobley[' SMILES'].apply(lambda x: MolToSmiles(MolFromSmiles(x), isomericSmiles=False, canonical=True))

In [35]:
mobley.shape

(643, 11)

In [37]:
mobley = mobley.drop_duplicates(subset='nonstereo_aromatic_smiles')
mobley.shape

(639, 11)

In [39]:
mobley = mobley.rename(columns={' experimental value (kcal/mol)': 'solubility'})

In [41]:
mobley = mobley[['nonstereo_aromatic_smiles', 'solubility']]

In [42]:
mobley.head()

,nonstereo_aromatic_smiles,solubility
0,CCCCCC(=O)OC,-2.49
1,CCCCO,-4.72
2,Clc1ccc(-c2cc(Cl)c(Cl)c(Cl)c2Cl)cc1Cl,-3.04
3,NC1CCCCC1,-4.59
4,O=COc1ccccc1,-3.82


In [43]:
mobley.to_csv('regression/solubility_mobley.csv', index=False)

In [44]:
test = pd.read_pickle('database.pickle')

In [48]:
pd.DataFrame(test).T

,calc_vdw,calc_h,d_vdw,PubChemID,d_expt,calc_reference,d_charging,d_calc_s (cal/mol.K),d_calc_h,h_solv,...,expt_h_reference,calc,calc_charging,notes,iupac,expt,expt_s (cal/K.mol),expt_h,d_expt_h,calc_s
mobley_1323538,2.131,-27.846859,0.031,6535,0.2,10.1021/jp0667442,0.019,2.366906,0.704722,-29.141947,...,Not available,-10.251,-12.382,[Experimental uncertainty as suggested by 10.1...,triethylphosphate,-7.5,Not available,Not available,Not available,NaN
mobley_2661134,0.378,-19.705391,0.025,13394,0.6,10.1021/ct800409d,0.017,2.371607,0.706458,-19.9761,...,Not available,-7.739,-8.117,[Experimental uncertainty not presently availa...,3-hydroxybenzonitrile,-9.65,Not available,Not available,Not available,NaN
mobley_6929123,1.301,-14.038273,0.022,7295,0.6,10.1021/ct800409d,0.01,2.330051,0.69429,-14.121598,...,Not available,-3.816,-5.117,[Experimental uncertainty not presently availa...,methyl 2-chloroacetate,-4.0,Not available,Not available,Not available,NaN
mobley_3395921,1.023,-13.12395,0.022,8894,0.6,10.1021/ct800409d,0.008,2.36006,0.703242,-13.284525,...,Not available,-1.809,-2.832,[Experimental uncertainty not presently availa...,tetrahydropyran,-3.12,Not available,Not available,Not available,NaN
mobley_5449201,2.664,-7.238288,0.023,8003,0.2,10.1021/jp0667442,0.0,2.348697,0.699886,-7.371649,...,Not available,2.673,0.009,[Experimental uncertainty as suggested by 10.1...,n-pentane,2.3,Not available,Not available,Not available,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
mobley_3775790,1.442,-13.453252,0.021,7304,0.6,10.1021/ct800409d,0.011,2.37866,0.708791,-13.523307,...,Not available,-2.374,-3.816,[Experimental uncertainty not presently availa...,1-methylpyrrole,-2.89,Not available,Not available,Not available,NaN
mobley_7157427,2.663,-10.525974,0.029,12371,0.6,10.1021/ct800409d,0.004,2.329902,0.694055,-10.714252,...,Not available,1.467,-1.196,[Experimental uncertainty not presently availa...,1-chloroheptane,0.29,Not available,Not available,Not available,NaN
mobley_3746675,2.512,-12.486237,0.027,7407,0.6,10.1021/ct800409d,0.009,2.371618,0.706503,-12.504832,...,Not available,-0.651,-3.163,[Experimental uncertainty not presently availa...,isopropenylbenzene,-1.24,Not available,Not available,Not available,NaN
mobley_3715043,1.575,-15.910346,0.032,11304,0.6,10.1021/ct800409d,0.011,2.369015,0.705551,-16.118075,...,Not available,-3.081,-4.656,[Experimental uncertainty not presently availa...,"1,4-dimethylnaphthalene",-2.82,Not available,Not available,Not available,NaN


In [52]:
sm = 'Cc1occc1C(=O)Nc1ccccc1'
sm = 'N#CC(OC1OC(COC2OC(CO)C(O)C(O)C2O)C(O)C(O)C1O)c1ccccc1'

mobley.loc[mobley.nonstereo_aromatic_smiles == sm]

,nonstereo_aromatic_smiles,solubility


In [53]:
delaney.loc[delaney.nonstereo_aromatic_smiles == sm]

,nonstereo_aromatic_smiles,solubility
382,N#CC(OC1OC(COC2OC(CO)C(O)C(O)C2O)C(O)C(O)C1O)c...,-0.77
